In [1]:
import os

In [2]:
%pwd

'c:\\Users\\jagat\\Downloads\\trial-2\\Text-Summarizer-Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\jagat\\Downloads\\trial-2\\Text-Summarizer-Project'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt = config.model_ckpt,
            num_train_epochs = params.num_train_epochs,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            weight_decay = params.weight_decay,
            logging_steps = params.logging_steps,
            evaluation_strategy = params.evaluation_strategy,
            eval_steps = params.evaluation_strategy,
            save_steps = params.save_steps,
            gradient_accumulation_steps = params.gradient_accumulation_steps
        )

        return model_trainer_config

In [8]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch
from torch import device
from accelerate import DataLoaderConfiguration

[2024-03-20 19:07:49,450: INFO: config: PyTorch version 2.2.1 available.]


In [17]:


class ModelTrainer:
    # def __init__(self, config: ModelTrainerConfig):
    #     self.config = config
    def __init__(self, config: ModelTrainerConfig, tokenizer):
        self.config = config
        self.tokenizer = tokenizer  # Store tokenizer in the class




    
    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        # tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        # model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        # seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_bart)
        # device = "cuda" if torch.cuda.is_available() else "cpu"

        # **Fine-tuned BART Support (replace with your fine-tuned model and tokenizer paths)**
        model_ckpt = "facebook/bart-base"  # Replace with actual path
        tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
        model_bart = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_bart)

        #loading data 
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        # trainer_args = TrainingArguments(
        #     output_dir=self.config.root_dir, num_train_epochs=self.config.num_train_epochs, warmup_steps=self.config.warmup_steps,
        #     per_device_train_batch_size=self.config.per_device_train_batch_size, per_device_eval_batch_size=self.config.per_device_train_batch_size,
        #     weight_decay=self.config.weight_decay, logging_steps=self.config.logging_steps,
        #     evaluation_strategy=self.config.evaluation_strategy, eval_steps=self.config.eval_steps, save_steps=1e6,
        #     gradient_accumulation_steps=self.config.gradient_accumulation_steps
        # )

        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir, num_train_epochs=1, warmup_steps=500,
            per_device_train_batch_size=1, per_device_eval_batch_size=1,
            weight_decay=0.01, logging_steps=10,
            evaluation_strategy='steps', eval_steps=500, save_steps=1e6,
            gradient_accumulation_steps=16
        )

        # trainer = Trainer(model=model_pegasus, args=trainer_args,
        #           tokenizer=tokenizer, data_collator=seq2seq_data_collator,
        #           train_dataset=dataset_samsum_pt["train"], 
        #           eval_dataset=dataset_samsum_pt["validation"])
        
        # trainer.train()

        # ## Save model
        # model_pegasus.save_pretrained(os.path.join(self.config.root_dir,"pegasus-samsum-model"))
        # ## Save tokenizer
        # tokenizer.save_pretrained(os.path.join(self.config.root_dir,"tokenizer"))
        
        trainer = Trainer(
            model=model_bart,  # Use the fine-tuned BART model here
            args=trainer_args,
            tokenizer=tokenizer,
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["train"],
            eval_dataset=dataset_samsum_pt["validation"]
        )

        trainer.train()

        # Save model and tokenizer
        model_bart.save_pretrained(os.path.join(self.config.root_dir, "bart-samsum-model"))
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))


In [18]:
# try:
#     config = ConfigurationManager()
#     model_trainer_config = config.get_model_trainer_config()
#     model_trainer_config = ModelTrainer(config=model_trainer_config)
#     model_trainer_config.train()
# except Exception as e:
#     raise e
# ... other code ...

try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()

    # **Define tokenizer outside**
    model_ckpt = "facebook/bart-base"  # Replace with actual path
    tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

    model_trainer = ModelTrainer(config=model_trainer_config, tokenizer=tokenizer)
    model_trainer.train()
except Exception as e:
    raise e


[2024-03-20 19:19:49,796: INFO: common: yaml file: config\config.yaml loaded successfully]
[2024-03-20 19:19:49,810: INFO: common: yaml file: params.yaml loaded successfully]
[2024-03-20 19:19:49,814: INFO: common: created directory at: artifacts]
[2024-03-20 19:19:49,818: INFO: common: created directory at: artifacts/model_trainer]


c:\Users\jagat\anaconda3\envs\textS\Lib\site-packages\accelerate\accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


  0%|          | 0/920 [00:00<?, ?it/s]

{'loss': 4.4038, 'grad_norm': 28.93683433532715, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.01}
{'loss': 4.0421, 'grad_norm': 22.651220321655273, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.02}
{'loss': 3.5716, 'grad_norm': 13.593120574951172, 'learning_rate': 3e-06, 'epoch': 0.03}
{'loss': 2.9027, 'grad_norm': 9.692885398864746, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.04}
{'loss': 2.4928, 'grad_norm': 7.944742202758789, 'learning_rate': 5e-06, 'epoch': 0.05}
{'loss': 2.3347, 'grad_norm': 7.999116897583008, 'learning_rate': 6e-06, 'epoch': 0.07}
{'loss': 2.2127, 'grad_norm': 7.614251613616943, 'learning_rate': 7.000000000000001e-06, 'epoch': 0.08}
{'loss': 2.2359, 'grad_norm': 8.661009788513184, 'learning_rate': 8.000000000000001e-06, 'epoch': 0.09}
{'loss': 2.1354, 'grad_norm': 7.732752799987793, 'learning_rate': 9e-06, 'epoch': 0.1}
{'loss': 2.0955, 'grad_norm': 10.624075889587402, 'learning_rate': 1e-05, 'epoch': 0.11}
{'loss': 2.0463, 'grad_norm': 7.3705

  0%|          | 0/818 [00:00<?, ?it/s]

{'eval_loss': 1.5828601121902466, 'eval_runtime': 274.03, 'eval_samples_per_second': 2.985, 'eval_steps_per_second': 2.985, 'epoch': 0.54}
{'loss': 1.8625, 'grad_norm': 6.273148536682129, 'learning_rate': 4.880952380952381e-05, 'epoch': 0.55}
{'loss': 1.7794, 'grad_norm': 5.473513126373291, 'learning_rate': 4.761904761904762e-05, 'epoch': 0.56}
{'loss': 1.8457, 'grad_norm': 5.096785545349121, 'learning_rate': 4.642857142857143e-05, 'epoch': 0.58}
{'loss': 1.7469, 'grad_norm': 6.7713117599487305, 'learning_rate': 4.523809523809524e-05, 'epoch': 0.59}
{'loss': 1.8842, 'grad_norm': 5.602489471435547, 'learning_rate': 4.404761904761905e-05, 'epoch': 0.6}
{'loss': 1.8695, 'grad_norm': 6.427999973297119, 'learning_rate': 4.2857142857142856e-05, 'epoch': 0.61}
{'loss': 1.8701, 'grad_norm': 5.662527084350586, 'learning_rate': 4.166666666666667e-05, 'epoch': 0.62}
{'loss': 1.8355, 'grad_norm': 6.835543155670166, 'learning_rate': 4.047619047619048e-05, 'epoch': 0.63}
{'loss': 1.7173, 'grad_norm'

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}


{'loss': 1.6696, 'grad_norm': 5.013625621795654, 'learning_rate': 0.0, 'epoch': 1.0}
{'train_runtime': 26034.0285, 'train_samples_per_second': 0.566, 'train_steps_per_second': 0.035, 'train_loss': 1.939726742454197, 'epoch': 1.0}
